In [6]:
import os
import tempfile
import shutil
from typing import List, Optional
from pathlib import Path
from settings import settings


from fastapi import FastAPI, UploadFile, File, HTTPException, Form
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import StreamingResponse
from pydantic_settings import BaseSettings
import json

# LlamaIndex Imports
from llama_index.core import Settings, StorageContext, VectorStoreIndex, SimpleDirectoryReader
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding
from llama_index.core.memory import ChatMemoryBuffer
import chromadb


In [10]:
# Initialize FastAPI
app = FastAPI(title=settings.app_config.app_name)

# Enable CORS (Allows React frontend on localhost:3000 to talk to this backend)
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],  # In production, replace "*" with specific domain
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

Step 1: Import UUID

In [ ]:
# Step 1: Import UUID
import uuid

# Step 2: Initialize the Session Store
# In-memory store for active user sessions
# Structure: { "session_id": { "chat_engine": engine_object, "filename": "doc.pdf" } }
active_sessions = {}

In [ ]:
def initialize_engine(api_key: str, pdf_path: str):
    """
    Creates a NEW chat engine for a specific user.
    Returns the engine object so we can store it in the session dictionary.
    """
    # 1. Setup Models (Specific to this request)
    # We use the settings you imported earlier
    llm = GoogleGenAI(model="models/gemini-2.5-flash", api_key=api_key)
    embed_model = GoogleGenAIEmbedding(model="models/gemini-embedding-001", api_key=api_key)
    
    # Apply settings locally for this index creation
    from llama_index.core import Settings as LlamaSettings
    LlamaSettings.llm = llm
    LlamaSettings.embed_model = embed_model

    # 2. Setup Persistent ChromaDB
    # We use the path from your settings.py
    persistent_client = chromadb.PersistentClient(path=settings.app_config.db_path)
    
    # CRITICAL CHANGE: Create a unique collection name per session? 
    # For now, we will use one shared collection but isolate via logic, 
    # OR better: Use a unique collection name per session to prevent data mixing.
    # Let's generate a unique collection name based on a timestamp or random ID to be safe.
    import time
    unique_collection_name = f"collection_{int(time.time())}" 
    
    chroma_collection = persistent_client.get_or_create_collection(unique_collection_name)
    vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
    storage_context = StorageContext.from_defaults(vector_store=vector_store)

    # 3. Load Documents
    documents = SimpleDirectoryReader(input_files=[pdf_path]).load_data()
    
    if not documents:
        raise ValueError("No text could be extracted from the PDF.")

    # 4. Create Index
    index = VectorStoreIndex.from_documents(
        documents, 
        storage_context=storage_context,
        show_progress=True
    )

    # 5. Create Chat Engine with Memory
    memory = ChatMemoryBuffer.from_defaults(token_limit=3900)
    
    chat_engine = index.as_chat_engine(
        chat_mode="context",
        memory=memory,
        system_prompt=(
            "You are a helpful assistant answering questions based strictly on the provided PDF context. "
            "If the answer is not in the context, say 'I cannot find this information in the document.' "
            "Always cite the page number if available."
        ),
    )
    
    # RETURN the engine instead of saving to a global variable
    return chat_engine, unique_collection_name

In [ ]:
@app.post("/upload")
async def upload_pdf(file: UploadFile = File(...), api_key: str = Form(...)):
    if not file.filename.endswith(".pdf"):
        raise HTTPException(status_code=400, detail="Only PDF files are allowed.")

    # 1. Generate a Unique Session ID for this user
    session_id = str(uuid.uuid4())
    
    # 2. Save uploaded file temporarily
    temp_dir = tempfile.mkdtemp()
    temp_file_path = os.path.join(temp_dir, file.filename)
    
    try:
        with open(temp_file_path, "wb") as buffer:
            shutil.copyfileobj(file.file, buffer)
        
        # 3. Initialize the engine (Get engine + collection name)
        chat_engine, collection_name = initialize_engine(api_key, temp_file_path)
        
        # 4. STORE in our dictionary
        active_sessions[session_id] = {
            "chat_engine": chat_engine,
            "collection_name": collection_name,
            "filename": file.filename
        }
        
        # 5. Return the Session ID to the user
        # The frontend MUST save this ID and send it with every chat request
        return {
            "message": "PDF processed successfully!", 
            "session_id": session_id,
            "filename": file.filename
        }
    
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Processing failed: {str(e)}")
    
    finally:
        # Cleanup temp file
        if os.path.exists(temp_file_path):
            os.remove(temp_file_path)
        if os.path.exists(temp_dir):
            os.rmdir(temp_dir)

In [ ]:
# @app.post("/chat")
# async def chat(session_id: str = Form(...), query: str = Form(...)):
#     # 1. Look up the session
#     if session_id not in active_sessions:
#         raise HTTPException(status_code=404, detail="Session not found. Please upload a PDF first.")
    
#     # 2. Get the specific engine for this user
#     session_data = active_sessions[session_id]
#     engine = session_data["chat_engine"]
    
#     # 3. Chat using THAT specific engine
#     response = engine.chat(query)
#     # ... return response

In [ ]:
@app.post("/chat")
async def chat(session_id: str = Form(...), query: str = Form(...)):
    # 1. VALIDATION: Check if the session exists in our dictionary
    if session_id not in active_sessions:
        raise HTTPException(
            status_code=404, 
            detail="Session not found. Please upload a PDF first to get a session_id."
        )
    
    # 2. RETRIEVAL: Get the specific data for this user
    session_data = active_sessions[session_id]
    engine = session_data["chat_engine"]
    
    # Optional: Update API key if the user sent a new one (flexibility)
    # But usually, the engine already has the key from the upload step.

    try:
        # 3. EXECUTION: Run the query on the SPECIFIC engine
        # This uses the memory stored inside this specific engine instance
        response = engine.chat(query)
        
        # 4. FORMATTING: Extract sources for the frontend to display
        sources = []
        if response.sources:
            for source in response.sources:
                # Safely get metadata, defaulting to 'N/A' if missing
                sources.append({
                    "text": source.node.text[:200] + "...", # Snippet preview
                    "page": source.node.metadata.get("page_label", "N/A"),
                    "file": source.node.metadata.get("file_name", "Unknown")
                })

        # 5. RETURN: Send structured JSON back to the frontend
        return {
            "answer": response.response,
            "sources": sources,
            "session_id": session_id # Echo back the ID for confirmation
        }
    
    except Exception as e:
        # Handle any LLM or retrieval errors gracefully
        raise HTTPException(status_code=500, detail=f"Chat failed: {str(e)}")

In [ ]:
@app.post("/chat/stream")
async def chat_stream(session_id: str = Form(...), query: str = Form(...)):
    # 1. VALIDATION (Same as above)
    if session_id not in active_sessions:
        raise HTTPException(status_code=404, detail="Session not found.")
    
    # 2. RETRIEVAL
    engine = active_sessions[session_id]["chat_engine"]

    # 3. GENERATOR FUNCTION: This runs asynchronously
    async def event_generator():
        try:
            # Use stream_chat instead of chat
            response = engine.stream_chat(query)
            
            # Iterate over tokens as they are generated
            for token in response.response_gen:
                # Format as Server-Sent Event (SSE)
                # The 'data: ' prefix is required by the SSE protocol
                data = json.dumps({"token": token})
                yield f"data: {data}\n\n"
            
            # After the text is done, send the sources as a final event
            sources = []
            if response.sources:
                for source in response.sources:
                    sources.append({
                        "page": source.node.metadata.get("page_label", "N/A"),
                        "snippet": source.node.text[:150]
                    })
            
            final_data = json.dumps({"sources": sources, "done": True})
            yield f"data: {final_data}\n\n"
            
        except Exception as e:
            # Send error as an SSE event so the frontend can show it
            error_data = json.dumps({"error": str(e)})
            yield f"data: {error_data}\n\n"

    # 4. RETURN STREAMING RESPONSE
    # media_type='text/event-stream' tells the browser "This is a live stream"
    return StreamingResponse(event_generator(), media_type="text/event-stream")